# Experiment C-1


In [1]:
"""C-1: Train NN with TensorFlow/PyTorch and evaluate TensorFlow logistic regression."""

import numpy as np


def load_data():
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    data = load_breast_cancer()
    X = data.data.astype(np.float32)
    y = data.target.astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    return X_train, X_test, y_train, y_test


def run_tensorflow_models(X_train, X_test, y_train, y_test):
    try:
        import tensorflow as tf
    except ImportError:
        print("TensorFlow not installed. Skipping TensorFlow section.")
        return

    print("\n=== TensorFlow Logistic Regression ===")
    log_reg = tf.keras.Sequential(
        [tf.keras.layers.Input(shape=(X_train.shape[1],)), tf.keras.layers.Dense(1, activation="sigmoid")]
    )
    log_reg.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()],
    )
    log_reg.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    results = log_reg.evaluate(X_test, y_test, verbose=0)
    print(f"Loss={results[0]:.4f}, Acc={results[1]*100:.2f}%, Precision={results[2]:.4f}, Recall={results[3]:.4f}, AUC={results[4]:.4f}")

    print("\n=== TensorFlow Neural Network ===")
    nn = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train.shape[1],)),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    nn.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    _, nn_acc = nn.evaluate(X_test, y_test, verbose=0)
    print(f"NN Test Accuracy: {nn_acc*100:.2f}%")


def run_pytorch_model(X_train, X_test, y_train, y_test):
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
    except ImportError:
        print("PyTorch not installed. Skipping PyTorch section.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32).to(device)
    x_te = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_te = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32).to(device)

    model = nn.Sequential(
        nn.Linear(X_train.shape[1], 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 1),
        nn.Sigmoid(),
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for _ in range(200):
        model.train()
        optimizer.zero_grad()
        out = model(x_tr)
        loss = criterion(out, y_tr)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = (model(x_te) >= 0.5).float()
        acc = (preds.eq(y_te).float().mean().item()) * 100
    print(f"\n=== PyTorch Neural Network ===\nNN Test Accuracy: {acc:.2f}%")


def main() -> None:
    X_train, X_test, y_train, y_test = load_data()
    run_tensorflow_models(X_train, X_test, y_train, y_test)
    run_pytorch_model(X_train, X_test, y_train, y_test)


if __name__ == "__main__":
    main()



ModuleNotFoundError: No module named 'numpy'